# 🚀 Google Gemini SDK: Beginner to Intermediate Guide

Welcome to the hands-on Google Gemini tutorial! This notebook will take you from sending your first API request to constructing an **Agent with Function Calling (Tool Use)**.

### Setup Prerequisites
Before starting, make sure you have installed the modern `google-genai` SDK and set your API key.

```bash
pip install google-genai pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# Load API key from environment or .env file
load_dotenv()

# Initialize the client (automatically uses GEMINI_API_KEY from environment)
client = genai.Client()

## Level 1: Basic Text Generation (Hello World)

**Concept:** The simplest way to interact with Gemini is by sending a single prompt to `client.models.generate_content()`.

We'll use `gemini-2.5-flash` as our default model for fast, standard tasks.

In [ ]:
# Level 1 Example: Basic Text Generation
response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Explain what an API is in 2 concise sentences."
)

print("--- Level 1 Response ---")
print(response.text)

## Level 2: System Instructions & Configurations

**Concept:** You can influence Gemini's personality, creativity, and tone using `GenerateContentConfig`.
- `system_instruction`: Dictates the persona or strict rules the model must follow.
- `temperature`: Controls randomness (0.0 = deterministic/factual, 1.0 = creative).

In [ ]:
# Level 2 Example: Custom System Prompt & Low Temperature
config = types.GenerateContentConfig(
    system_instruction="You are a strict, highly accurate math and logic tutor. Be concise.",
    temperature=0.1
)

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents="Is 107 a prime number? Show quick logic.",
    config=config
)

print("--- Level 2 Response ---")
print(response.text)

## Level 3: Enforcing Structured JSON Output

**Concept:** Instead of unstructured text, you can enforce Gemini to output pure, type-safe JSON schema using Python's `pydantic`.

In [ ]:
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# -------------------------------------------------------------------------
# Step 1: Define your target output structure using Pydantic
# -------------------------------------------------------------------------
# Inheriting from BaseModel tells Pydantic that this class represents a structured data model.
class Flashcard(BaseModel):
    # Field(description=...) gives Gemini semantic context so it knows what to populate in each field.
    term: str = Field(description="The word or concept name")
    definition: str = Field(description="A 1-sentence easy definition")
    example: str = Field(description="A real-world example")


# -------------------------------------------------------------------------
# Step 2: Configure Gemini's output generation behavior
# -------------------------------------------------------------------------
json_config = types.GenerateContentConfig(
    # Sets the output format to JSON instead of default free-form plain text/markdown.
    response_mime_type="application/json",
    
    # Enforces strict adherence to our Pydantic schema structure.
    # Gemini will dynamically convert this Python class into a JSON Schema.
    response_schema=Flashcard,
    
    # Controls answer randomness: 0.0 is deterministic and factual; higher values (e.g., 0.8) are more creative.
    temperature=0.2,
    
    # Explicitly disables the Automatic Function Calling (AFC) handler in the SDK,
    # preventing internal warnings when generating structured schemas without calling functions.
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
)


# -------------------------------------------------------------------------
# Step 3: Execute the request
# -------------------------------------------------------------------------
response = client.models.generate_content(
    # Specifies which model variant to run (e.g., gemini-2.5-flash or gemini-3.5-flash-lite).
    model="gemini-3.5-flash",
    
    # The prompt containing the text instructions for Gemini to analyze.
    contents="Generate a study flashcard for the concept of 'Recursion'.",
    
    # Applies our defined JSON schema and temperature configurations to this execution.
    config=json_config
)

# Print the structured JSON string returned by Gemini
print("--- Level 3 Output (Structured JSON) ---")
print(response.text)

## Level 4: Multi-Turn Chat (Conversational Memory)

**Concept:** Instead of manually tracking conversation history, use `client.chats.create()`. The chat object automatically remembers prior messages.

In [ ]:
# Level 4 Example: Starting a chat session
chat = client.chats.create(model="gemini-3.5-flash")

# Message 1
response_1 = chat.send_message("My favorite programming language is Python.")
print("Bot:", response_1.text)

# Message 2 (Gemini remembers the context)
response_2 = chat.send_message("What is my favorite language?")
print("\nBot:", response_2.text)

## Level 5 (Intermediate): Tool Use / Function Calling

**Concept:** LLMs cannot execute code or access live data directly. However, Gemini can detect when it needs external information, pause execution, and request your code to execute a Python function on its behalf!

Using Automatic Function Calling (AFC) in `client.chats`, Gemini will automatically call local Python functions and weave their return values back into its final response.

In [ ]:
# 1. Define a standard Python function with type hints and docstrings
def get_stock_price(symbol: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    
    Args:
        symbol: The stock ticker symbol (e.g., GOOGL, AAPL).
    """
    # Mocking database / external API call
    mock_db = {
        "GOOGL": "$185.50",
        "AAPL": "$225.10",
        "MSFT": "$450.00"
    }
    symbol_upper = symbol.upper()
    price = mock_db.get(symbol_upper, "Price unavailable")
    return f"The current price for {symbol_upper} is {price}."

# 2. Pass the function directly into the chat configuration's `tools` array
agent_chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        tools=[get_stock_price],  # Registering our python function as a tool
        temperature=0.0
    )
)

# 3. Query the agent with a question that requires external data lookup
print("--- Level 5 Agent Processing ---")
user_query = "Hey, what is the current price of AAPL stock?"
response = agent_chat.send_message(user_query)

print("Final Agent Response:")
print(response.text)

# Lesson 5 (Part 2 with Tavily)
```bash
pip install google-genai tavily-python

In [ ]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Define the Python Tool Function
# -------------------------------------------------------------------------
def search_web(query: str) -> dict:
    """Performs a live web search using Tavily to retrieve current news and information.
    
    Args:
        query: The search query string (e.g., 'latest AI breakthroughs this week').
    """
    print(f"\n[AFC TRIGGERED] Executing Tavily search for query: '{query}'...")
    
    # Run Tavily search returning clean text snippets
    response = tavily_client.search(query=query, search_depth="basic", max_results=3)
    
    # Structure the extracted fields for Gemini
    results = [
        {
            "title": r.get("title"),
            "url": r.get("url"),
            "content": r.get("content")
        }
        for r in response.get("results", [])
    ]
    return {"results": results}

# -------------------------------------------------------------------------
# Step 2: Initialize Chat with AFC Enabled
# -------------------------------------------------------------------------
# Passing `tools=[search_web]` to a chat object enables Automatic Function Calling (AFC).
# The SDK handles the multi-turn loop automatically.
agent_chat = client.chats.create(
    model="gemini-3.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are a real-time news and research assistant. Use the search_web tool "
            "to look up current events and live data. Always include source URLs in your answer."
        ),
        # Pass the python function directly as a tool
        tools=[search_web],
        temperature=0.0
    )
)


# -------------------------------------------------------------------------
# Step 3: Run the Query
# -------------------------------------------------------------------------
user_prompt = "What are the latest key updates on AI agent frameworks?"
print(f"User Query: {user_prompt}")

# Send message: Gemini detects search intent, calls search_web(), and returns the final text response.
response = agent_chat.send_message(user_prompt)

print("\n--- Final Agent Response (with Source Citations) ---")
print(response.text)

## Level 6 (Intermediate): Manual Function Calling with Tavily Web Search

**Concept:** Automatic Function Calling (AFC) is great for rapid prototyping, but in production systems you often want **Manual Function Calling**. 

Manual tool calling gives you full control over the execution loop:
1. Gemini decides *if* and *which* tool to invoke, returning a `function_call` payload.
2. Your Python app intercepts the request, runs the local function (e.g., querying Tavily), and validates the results.
3. Your app manually sends the `function_response` back to Gemini to generate the final human-readable answer.

---

### Step 1: Install Tavily and Setup Keys

```bash
pip install google-genai tavily-python python-dotenv

In [ ]:
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

In [ ]:
# -------------------------------------------------------------------------
# Step 1: Define the Tavily Search Tool Function
# -------------------------------------------------------------------------
def search_web(query: str) -> dict:
    """Performs a web search using Tavily to get real-time search results.
    
    Args:
        query: The search query string.
    """
    print(f"\n[LOCAL EXECUTION] Running Tavily search for: '{query}'...")
    response = tavily_client.search(query=query, search_depth="basic", max_results=3)
    
    # Extract titles, URLs, and text content for Gemini
    results = [
        {"title": r.get("title"), "url": r.get("url"), "content": r.get("content")}
        for r in response.get("results", [])
    ]
    return {"results": results}

# Map function names to their Python callable reference
available_tools = {
    "search_web": search_web
}

In [ ]:
# -------------------------------------------------------------------------
# Step 2: Configure Gemini with Tool Declarations & Disable AFC
# -------------------------------------------------------------------------
config = types.GenerateContentConfig(
    system_instruction="You are a real-time web researcher. Use search_web to answer live queries.",
    tools=[search_web],
    temperature=0.0,
    # Crucial: Disable AFC so Gemini stops execution and returns the tool call object to python
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
)

# -------------------------------------------------------------------------
# Step 3: Turn 1 — Send User Prompt to Gemini
# -------------------------------------------------------------------------
user_prompt = "What are the latest developments in AI agents this week?"
print(f"User Query: {user_prompt}")

# Store the conversation history explicitly
contents = [
    types.Content(
        role="user",
        parts=[types.Part.from_text(text=user_prompt)]
    )
]

# Send turn 1 to Gemini
response_turn1 = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=contents,
    config=config
)

In [ ]:
# -------------------------------------------------------------------------
# Step 4: Detect and Manually Execute Tool Call
# -------------------------------------------------------------------------
# Append model's response to the conversation history
contents.append(response_turn1.candidates[0].content)

# Check if Gemini requested a function call
function_calls = response_turn1.function_calls

if function_calls:
    for call in function_calls:
        function_name = call.name
        function_args = call.args
        
        print(f"\n[GEMINI PROPOSAL] Gemini requested tool: '{function_name}'")
        print(f"[GEMINI ARGS] {function_args}")
        
        # 1. Look up and execute the local python function manually
        if function_name in available_tools:
            tool_result = available_tools[function_name](**function_args)
            
            # 2. Append the function output back into conversation history as a 'function_response' part
            contents.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_function_response(
                            name=function_name,
                            response=tool_result
                        )
                    ]
                )
            )

# -------------------------------------------------------------------------
# Step 5: Turn 2 — Pass Tool Results back to Gemini for Final Answer
# -------------------------------------------------------------------------
response_turn2 = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=contents,
    config=config
)

print("\n--- Final Answer from Gemini ---")
print(response_turn2.text)